# Model B (Backend) — Terrain Segmentation + Sponge-Zone Detection + GeoJSON

This notebook operationalizes the **Model B pipeline** for local/backend execution in a way that is **compatible with the CHRIS FastAPI backend**. It intentionally uses the same Python functions that the API uses (`app/ml/model_b.py`) so that outputs are consistent between offline runs and production inference.

This notebook covers:

1. Loading exported Model B artifacts (ONNX + `meta.json`).
2. Running U-Net segmentation inference on an RGB image patch.
3. Detecting sponge zones (connected components over `vacant=1` OR `flooded=5`).
4. Generating an app-compatible GeoJSON `FeatureCollection` (pixel-coordinate polygons).
5. Writing outputs to disk (mask, zone metrics JSON, GeoJSON, optional visualization images).

It does **not** replicate Colab-only steps such as Copernicus downloads and training. Those steps are part of model development, while this notebook is for **reproducible backend inference**.

## Important limitations

The backend currently accepts a single 256×256 image patch without georeferencing metadata. Therefore, the GeoJSON geometry returned by this pipeline is expressed in **image pixel coordinates**, not lat/lon (WGS84). If you need georeferenced GeoJSON, you must provide upstream raster geotransform/CRS and convert pixel coordinates to map coordinates.


## 1) Environment setup

This notebook is designed to run inside the backend repo (`chris_backend`). The backend dependencies are in `requirements.txt`.

Recommended setup:

```bash
cd chennai-hydro-resilience-platform-313260-313270/chris_backend
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

Optional (only if you want inline plots):

```bash
pip install matplotlib
```


In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import asdict
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image


## 2) Import the backend Model B pipeline

The FastAPI app imports Model B functions from `app/ml/model_b.py`. This notebook does the same.

Because notebooks often run from a subdirectory, we first ensure the backend root is on `sys.path`.


In [ ]:
import sys

NOTEBOOK_DIR = Path().resolve()
backend_root = NOTEBOOK_DIR
while backend_root != backend_root.parent and not (backend_root / "app").exists():
    backend_root = backend_root.parent

if not (backend_root / "app").exists():
    raise RuntimeError(
        "Could not locate backend root (directory containing 'app/'). "
        "Run this notebook from within the chris_backend folder or its subfolders."
    )

sys.path.insert(0, str(backend_root))
print("Backend root:", backend_root)

from app.ml.model_b import (
    ModelBArtifacts,
    SpongeZoneConstants,
    detect_sponge_zones,
    load_model_b_onnx,
    predict_segmentation,
    sponge_zones_to_geojson,
)


## 3) Provision Model B artifacts (exported from Colab)

The backend expects Model B artifacts under:

```text
chris_backend/
  models/
    model_b/
      unet.onnx
      meta.json
```

The notebook (and API) use `meta.json` to determine the input size and optional sponge-zone constants.

### Minimum viable `meta.json`

At minimum, the backend loader reads:

- `input.shape` (defaults to `[256,256,3]` if omitted)
- `classes` (optional mapping of class IDs to human-readable names)
- `sponge_zones` (optional constants override)

An example `meta.json` compatible with this repo is documented in `chris_backend/README.md`.


In [ ]:
# Update these paths to match your local artifact location.
# If you follow the backend README, the defaults below should work.

MODEL_B_DIR = backend_root / "models" / "model_b"
ONNX_PATH = MODEL_B_DIR / "unet.onnx"
META_PATH = MODEL_B_DIR / "meta.json"

print("Model B directory:", MODEL_B_DIR)
print("ONNX exists:", ONNX_PATH.exists(), "->", ONNX_PATH)
print("meta.json exists:", META_PATH.exists(), "->", META_PATH)

if not ONNX_PATH.exists() or not META_PATH.exists():
    raise FileNotFoundError(
        "Model B artifacts not found. Place exported files at:\n"
        f"  {ONNX_PATH}\n"
        f"  {META_PATH}\n\n"
        "See chris_backend/README.md for Colab export steps and expected meta.json structure."
    )


## 4) Load artifacts

This uses the same artifact loader that the API uses: `load_model_b_onnx()`.


In [ ]:
art: ModelBArtifacts = load_model_b_onnx(str(ONNX_PATH), str(META_PATH))

print("Loaded Model B provider:", art.provider)
print("Input size:", art.meta.input_size)
print("Classes mapping present:", art.meta.classes is not None)
print("Sponge-zone constants:")
print(asdict(art.meta.sponge_constants))


## 5) Provide an input image patch

The backend pipeline expects an RGB image (PNG/JPG). It will be resized to `meta.input_size×meta.input_size` and normalized using the notebook’s authoritative logic (`uint8 / 255.0`).

To match app behavior, you should provide a 256×256 Sentinel-2 RGB patch (or a patch that your U-Net was trained to accept).


In [ ]:
# Set this to an RGB image patch you want to analyze.
# Suggested: put a file under chris_backend/notebooks/data/patch.png
INPUT_IMAGE_PATH = backend_root / "notebooks" / "data" / "sample_patch.png"

print("Input image exists:", INPUT_IMAGE_PATH.exists(), "->", INPUT_IMAGE_PATH)

if not INPUT_IMAGE_PATH.exists():
    raise FileNotFoundError(
        "Input image not found. Create a file at:\n"
        f"  {INPUT_IMAGE_PATH}\n\n"
        "The file should be an RGB PNG/JPG."
    )

image_bytes = INPUT_IMAGE_PATH.read_bytes()
img_preview = Image.open(INPUT_IMAGE_PATH).convert("RGB")
print("Preview size:", img_preview.size)
img_preview


## 6) Run segmentation inference

This executes the same logic as the backend endpoint `POST /predict/segmentation`:

- preprocess: resize to `input_size`, convert to RGB, normalize `/255.0`
- ONNX inference
- postprocess: `argmax` over per-pixel softmax logits


In [ ]:
seg_out: dict[str, Any] = predict_segmentation(art, image_bytes)

mask_list = seg_out["mask"]
proportions = seg_out["proportions"]

mask = np.array(mask_list, dtype=np.int32)
print("Mask shape:", mask.shape, "dtype:", mask.dtype)
print("Unique classes in mask:", np.unique(mask).tolist())
print("Proportions (by class id):", json.dumps(proportions, indent=2))


### Optional: visualize the segmentation

This uses the class-to-color mapping from the Model B Colab pipeline.

If you do not have `matplotlib`, the notebook will still run; the visualization here uses only Pillow.


In [ ]:
CLASS_COLORS = {
    0: (0, 0, 0),
    1: (255, 165, 0),
    2: (0, 128, 0),
    3: (0, 0, 255),
    4: (128, 128, 128),
    5: (0, 255, 255),
}

def mask_to_rgb(mask_2d: np.ndarray) -> np.ndarray:
    h, w = mask_2d.shape
    out = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in CLASS_COLORS.items():
        out[mask_2d == cls] = np.array(color, dtype=np.uint8)
    return out

mask_rgb = mask_to_rgb(mask)
Image.fromarray(mask_rgb)


## 7) Sponge-zone detection

This runs the same algorithm as the backend endpoint `POST /predict/terrain_analysis`:

1. Create a recharge candidate mask: `(vacant == 1) OR (flooded == 5)`.
2. Connected component labeling.
3. Filter by `min_zone_pixels`.
4. Compute area and volumetric capacity (depth-based).
5. Priority score: `area_ha * (2 if has_flood else 1)`.
6. Filter out unrealistic zones using `max_zone_ha`.


In [ ]:
zones_out: dict[str, Any] = detect_sponge_zones(
    mask=mask,
    classes=art.meta.classes,
    constants=art.meta.sponge_constants,
)

sponge_zones = zones_out["sponge_zones"]
sponge_summary = zones_out["sponge_summary"]

print("Detected zones:", len(sponge_zones))
print("Summary:\n", json.dumps(sponge_summary, indent=2))

# Show the top few zones
for z in sponge_zones[:5]:
    print(json.dumps(z, indent=2))


## 8) GeoJSON output

This generates the same GeoJSON payload as the backend endpoint `POST /predict/sponge_zones_geojson`.

The geometry is a **pixel-coordinate bounding-box polygon** around each connected component.


In [ ]:
geojson = sponge_zones_to_geojson(sponge_zones)
print("GeoJSON type:", geojson.get("type"))
print("Features:", len(geojson.get("features", [])))
geojson["features"][0] if geojson.get("features") else geojson


## 9) Write outputs to disk

This notebook writes a self-contained output bundle that is easy to consume from the app or during backend validation.

Outputs include:

- `segmentation_mask.json` (2D list of class IDs)
- `proportions.json`
- `sponge_zones.json`
- `sponge_summary.json`
- `sponge_zones.geojson`
- `mask_rgb.png` (optional visualization)
- `overlay.png` (optional visualization)


In [ ]:
run_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
OUT_DIR = backend_root / "notebooks" / "outputs" / f"model_b_run_{run_id}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def write_json(path: Path, obj: Any) -> None:
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")

write_json(OUT_DIR / "segmentation_mask.json", mask_list)
write_json(OUT_DIR / "proportions.json", proportions)
write_json(OUT_DIR / "sponge_zones.json", sponge_zones)
write_json(OUT_DIR / "sponge_summary.json", sponge_summary)
write_json(OUT_DIR / "sponge_zones.geojson", geojson)

# Save visualization images
mask_rgb_img = Image.fromarray(mask_rgb)
mask_rgb_img.save(OUT_DIR / "mask_rgb.png")

input_resized = Image.open(INPUT_IMAGE_PATH).convert("RGB").resize((art.meta.input_size, art.meta.input_size))
input_arr = np.array(input_resized, dtype=np.uint8)
overlay = (0.6 * input_arr + 0.4 * mask_rgb).astype(np.uint8)
Image.fromarray(overlay).save(OUT_DIR / "overlay.png")

print("Wrote outputs to:", OUT_DIR)
print("Files:")
for p in sorted(OUT_DIR.iterdir()):
    print(" -", p.name)


## 10) App compatibility notes

This notebook is designed to match the behavior of the backend endpoints in `app/routers/predict.py`:

1. `POST /predict/segmentation` maps to `predict_segmentation()`.
2. `POST /predict/terrain_analysis` maps to `predict_segmentation()` + `detect_sponge_zones()`.
3. `POST /predict/sponge_zones_geojson` maps to `predict_segmentation()` + `detect_sponge_zones()` + `sponge_zones_to_geojson()`.

If your offline outputs differ from the API outputs for the same input image, the most common root causes are:

1. Different preprocessing (e.g., using a different resize method or normalization).
2. ONNX export mismatch (e.g., channel order or output tensor shape not equal to `(1,H,W,C)`).
3. Different sponge-zone constants (pixel resolution, depth, min pixels, max hectares) due to differences in `meta.json`.
